# 03 · Core grid (H1, H2b): S0, S1, C0, C1 × 5 budgets × 5 seeds (+ exploratory 1%)

Runs on **Kaggle** (preferred: two T4s run two jobs in parallel) or Colab. The platform is detected automatically.

### Kaggle, one-time setup
1. Upload the preprocessed cache as a **private Kaggle Dataset**. Use the 7 files from Drive
   `MyDrive/robust-ecg-ssl/cache/`: `ptbxl_X.npy`, `ptbxl_meta.csv`, `ptbxl_statements.json`,
   `scp_statements.csv`, `norm_ptbxl_folds1-8.json`, `sph_X.npy`, `sph_meta.csv`.
2. In this notebook: **Add Input** → that dataset. Settings: **Accelerator: GPU T4 x2**, **Internet: on**.

### Running
- Use **Save Version → Save & Run All (Commit)**. It runs in the background (up to 12 h, so closing the browser is
  fine), and `/kaggle/working/work.tar` plus `results_export.zip` are saved as the version's output.
  The whole grid is estimated at ≈ 4 h on two T4s.
- **To continue after an interrupted or timed-out version:** Add Input → *Your Work* → this notebook's previous
  version output. The setup cell restores `work.tar`, so finished runs are skipped and unfinished ones restart
  (at most one run per GPU is repeated).
- If a run fails, the queue stops and prints the end of its log. Paste that into the chat.

Every run records `platform` and `gpu` in its `config.json` (D32). Do not use *Save a copy in GitHub* on Colab.

In [ ]:
import os, glob, shutil, zipfile, json, subprocess, sys, tarfile, hashlib
ON_KAGGLE = os.path.isdir('/kaggle/working')
if ON_KAGGLE:
    REPO, CACHE, WORK = '/tmp/robust-ecg-ssl', '/tmp/cache', '/tmp/work'
    ARCHIVE, EXPORT = '/kaggle/working/work.tar', '/kaggle/working/results_export.zip'
    hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ptbxl_X.npy', recursive=True)
            if os.path.exists(os.path.join(os.path.dirname(p), 'sph_X.npy'))]
    if not hits:
        raise FileNotFoundError('No input dataset with ptbxl_X.npy and sph_X.npy: add the cache dataset (see above).')
    CACHE_SRC, remount = hits[0], None
    prior = sorted(glob.glob('/kaggle/input/**/work.tar', recursive=True))
    if prior and not os.path.exists(WORK):
        with tarfile.open(prior[-1]) as t:
            t.extractall('/tmp')
        print('restored previous output from', prior[-1])
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/robust-ecg-ssl'
    REPO, CACHE, WORK = '/content/robust-ecg-ssl', '/content/cache', PROJECT      # WORK holds runs/ and ssl/
    ARCHIVE, EXPORT = None, f'{PROJECT}/results_export.zip'
    CACHE_SRC, remount = f'{PROJECT}/cache', (lambda: drive.mount('/content/drive', force_remount=True))
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Salako07/robust-ecg-ssl.git', REPO], check=True)
subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, f'{REPO}/src')
from robust_ecg.colab_utils import copy_dir_with_retry
copy_dir_with_retry(CACHE_SRC, CACHE, remount=remount)
import torch
GPUS = max(1, torch.cuda.device_count())
print('platform:', 'kaggle' if ON_KAGGLE else 'colab', '| GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Cache fingerprint
SHA-256 of every cache file, so we can confirm the Kaggle copy is byte-identical to the one on Drive (~30 s).

In [ ]:
def sha(p, bs=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while (b := f.read(bs)):
            h.update(b)
    return h.hexdigest()
manifest = {f: dict(bytes=os.path.getsize(f'{CACHE}/{f}'), sha256=sha(f'{CACHE}/{f}')) for f in sorted(os.listdir(CACHE))}
json.dump(manifest, open(f'{WORK}/cache_manifest.json', 'w'), indent=1)
for f, m in manifest.items():
    print(f'{f:28s} {m["bytes"]:>14,d}  {m["sha256"][:16]}')

## Run the queue
One line per finished run. Per-run console logs are in `work/logs/`.

In [ ]:
cmd = [sys.executable, f'{REPO}/scripts/run_queue.py', '--cache', CACHE, '--out', WORK, '--gpus', str(GPUS), '--workers', '2']
if ARCHIVE:
    cmd += ['--archive', ARCHIVE]
subprocess.run(cmd + ['--dry-run'], check=True)
!{' '.join(cmd)}

## Progress table
Mean over finished seeds. **Descriptive only**: the pre-registered analysis (paired patient-level bootstrap, Holm)
is a separate script.

In [ ]:
import pandas as pd
reg = pd.read_csv(f'{WORK}/runs/registry.csv')
for col in ('test_macro_auroc', 'sph_e3_macro_auroc', 'e3_degradation'):
    t = reg.pivot_table(index='budget', columns='arm', values=col, aggfunc=['mean', 'count'])
    print(col); print(t.round(4).to_string()); print()

## Export results for the repository
Small files only (`config.json`, `log.csv`, `done.json`, `predictions.npz`, registries, logs, cache manifest).
On Kaggle it is in the version's **Output** tab; on Colab it is on Drive. Download it and attach it in the chat.

In [ ]:
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ('runs', 'ssl'):
        for f in glob.glob(f'{WORK}/{sub}/*/*'):
            if f.endswith(('config.json', 'log.csv', 'done.json', 'predictions.npz')) and \
               os.path.exists(os.path.join(os.path.dirname(f), 'done.json')):
                z.write(f, os.path.relpath(f, WORK))
    for f in [f'{WORK}/runs/registry.csv', f'{WORK}/ssl/registry_ssl.csv', f'{WORK}/cache_manifest.json',
              *glob.glob(f'{WORK}/logs/*.log')]:
        if os.path.exists(f):
            z.write(f, os.path.relpath(f, WORK))
print(EXPORT, round(os.path.getsize(EXPORT) / 1e6, 2), 'MB,', len(zipfile.ZipFile(EXPORT).namelist()), 'files')